# Compare FAISS Search Methods: Model x Dataset

Every search method benchmarked by `benchmark_ann.py` (Exact FlatIP vs
HNSW vs IVF-PQ vs LSH) on the full 2x2 grid of embedding model x dataset,
all with the same protocol:

```text
1,000 held-out query vectors (seed 42)
        |
gallery vectors -> Exact FlatIP / HNSW / IVF-PQ / LSH
        |
exact top-K = ground truth -> Recall / Precision / mAP @5/10/20
```

| Config | Model | Dataset | Dim | Gallery |
|---|---|---|---:|---:|
| `vit-coco` | ViT-B/16 | COCO unlabeled2017 | 768 | 122,403 |
| `resnet-coco` | ResNet18 | COCO unlabeled2017 | 512 | 122,403 |
| `vit-zappos` | ViT-B/16 | UT-Zap50K | 768 | 49,025 |
| `resnet-zappos` | ResNet18 | UT-Zap50K | 512 | 49,025 |

Both models on a dataset are searched with the **identical 1,000
queries**, so differences between them come from the model alone.

Sections:

1. combined metrics table,
2. overlaid trade-off plots and parameter sweeps,
3. cross-configuration summary (speed-up / compression vs exact),
4. **model vs dataset effects** - the same method across the 2x2 grid,
5. the same query searched by both models and every method,
6. Zappos category / subcategory precision for both models,
7. sanity checks.

Sections 1-4 need only `results/<config>/ann_results.json`. Sections 5-7
also need a local `benchmark_ann.py` run (`per_query/`), the embeddings
and (for 5-6) the images; anything missing is skipped with a note.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Configurations to include (keys from datasets.py).
CONFIG_KEYS = ["vit-coco", "resnet-coco", "vit-zappos", "resnet-zappos"]

# Query image ids to visualise in section 5, per dataset. None = first 3
# of the 1,000 benchmark queries (shared by both models).
QUERY_IDS = {"coco": None, "zappos": None}

# Model whose ANN results are shown in the section 5 grids (both models'
# exact results are always shown).
GRID_MODEL = "vit"

# ANN settings shown in section 5 (file stems from
# results/<config>/per_query/, without "_top20.npy").
SHOW_SETTINGS = {
    "hnsw_ef128": "HNSW (efSearch=128)",
    "ivfpq_np32": "IVF-PQ (nprobe=32)",
    "lsh_nb512": "LSH (nbits=512)",
}

K_DISPLAY = 10   # results shown per method (metrics computed at this K)
SAVE_IMAGES = True

In [ ]:
# Find this experiment folder whether the notebook runs from its own
# folder or from the repository root, then reuse the shared registry.
EXPERIMENT_DIR = Path.cwd()
if not (EXPERIMENT_DIR / "datasets.py").exists():
    EXPERIMENT_DIR = Path.cwd() / "experiments" / "ann-comparison"
sys.path.insert(0, str(EXPERIMENT_DIR))
from datasets import CONFIGS, DATASETS, MODELS, REPO_ROOT, results_dir

runs = {}
for key in CONFIG_KEYS:
    path = results_dir(key) / "ann_results.json"
    if not path.exists():
        print(f"{key}: no ann_results.json - run benchmark_ann.py --config {key}")
        continue
    data = json.loads(path.read_text())
    cfg = CONFIGS[key]
    runs[key] = {
        "cfg": cfg,
        "label": cfg["label"],
        "model": cfg["model"],
        "dataset": cfg["dataset"],
        "metadata": data["metadata"],
        "rows": data["results"],
        "per_query_dir": results_dir(key) / "per_query",
        "output_dir": results_dir(key) / "visual-comparisons",
    }

for key, run in runs.items():
    m = run["metadata"]
    print(f"{run['label']:<20} gallery {m['gallery_size']:>7,} | queries {m['num_queries']} "
          f"| {m['dimension']}-d | k_gt {m['k_gt']} | seed {m['seed']}")

In [ ]:
# Images and labels are per dataset (shared by both models).
def load_images(dataset):
    image_dir = DATASETS[dataset]["image_dir"]
    if not image_dir.exists():
        return None
    if dataset == "coco":
        # COCO filenames are 12-digit zero-padded ids.
        return {"path": lambda image_id: image_dir / f"{int(image_id):012d}.jpg",
                "label": lambda image_id: "", "categories": None, "subcategories": None}
    # UT-Zap50K: ids are rows of the sorted image paths (same ordering as
    # pipeline/extract_embeddings.py).
    paths = sorted(image_dir.rglob("*.jpg"), key=lambda p: str(p))
    categories = np.array([p.relative_to(image_dir).parts[0] for p in paths])
    subcategories = np.array([p.relative_to(image_dir).parts[1] for p in paths])
    return {"path": lambda image_id: paths[int(image_id)],
            "label": lambda image_id: f"{categories[image_id]} / {subcategories[image_id]}",
            "categories": categories, "subcategories": subcategories}

images = {d: load_images(d) for d in sorted({run["dataset"] for run in runs.values()})}

# Per-query detail (sections 5-7): needs a local benchmark run plus the
# embeddings.
details = {}
for key, run in runs.items():
    cfg = run["cfg"]
    pq_dir = run["per_query_dir"]
    if not (pq_dir / "exact_top20.npy").exists():
        print(f"{run['label']}: per-query sections skipped - no per_query/ files")
        continue
    if not cfg["emb"].exists():
        print(f"{run['label']}: per-query sections skipped - embeddings missing: {cfg['emb']}")
        continue
    ids = np.load(cfg["ids"])
    query_rows = np.load(pq_dir / "query_rows.npy")
    details[key] = {
        "embeddings": np.load(cfg["emb"], mmap_mode="r"),
        "ids": ids,
        "query_rows": query_rows,
        "gallery_rows": np.load(pq_dir / "gallery_rows.npy"),
        "exact_top20": np.load(pq_dir / "exact_top20.npy"),
        "query_ids": ids[query_rows],
    }

for dataset, info in images.items():
    status = "found" if info else f"missing ({DATASETS[dataset]['image_dir']})"
    print(f"{DATASETS[dataset]['label']} images: {status}")
for key, d in details.items():
    print(f"{runs[key]['label']}: per-query ready | first query ids {d['query_ids'][:8].tolist()}")

## 1. Combined comparison

All accuracy metrics treat the **exact top-K of the same configuration**
as the relevant set:

- **Recall@K** = fraction of the exact top-K that the method also returns.
- **Precision@K** = fraction of the method's top-K that is in the exact
  top-K. Both lists hold K items, so it equals Recall@K here.
- **mAP@K** = mean over queries of AP@K = (1/K) * sum of Precision@i at
  every rank i holding a relevant result. Rank-aware: a method that finds
  the right neighbours but orders them wrongly scores lower.

All searches ran single-threaded (BLAS pinned to one thread); index builds
used all cores. All four configurations ran on the same machine.

In [ ]:
def setting_str(row):
    params = row["parameters"]
    algo = row["algorithm"]
    if algo == "HNSW":
        return f"efSearch={params['efSearch']}"
    if algo == "IVF-PQ":
        return f"nprobe={params['nprobe']}"
    if algo == "LSH":
        return f"nbits={params['nbits']}"
    return "-"

def metric(row, field, width=6):
    # Results from older runs may lack precision/mAP.
    value = None if row is None else row.get(field)
    return f"{'n/a':>{width}}" if value is None else f"{value:>{width}.3f}"

header = (
    f"{'Config':<20} {'Method':<13} {'Setting':<14} {'R@5':>6} {'R@10':>6} {'R@20':>6} "
    f"{'P@10':>6} {'mAP@5':>6} {'mAP@10':>6} {'mAP@20':>6} "
    f"{'ms/query':>9} {'QPS':>8} {'build(s)':>9} {'size(MB)':>9}"
)
print(header)
print("-" * len(header))
for key, run in runs.items():
    for row in run["rows"]:
        print(
            f"{run['label']:<20} {row['algorithm']:<13} {setting_str(row):<14} "
            f"{row['recall_at_5']:>6.3f} {row['recall_at_10']:>6.3f} {row['recall_at_20']:>6.3f} "
            f"{metric(row, 'precision_at_10')} {metric(row, 'map_at_5')} "
            f"{metric(row, 'map_at_10')} {metric(row, 'map_at_20')} "
            f"{row['latency_ms_per_query']:>9.3f} {row['queries_per_second']:>8.1f} "
            f"{row['build_seconds']:>9.1f} {row['index_size_mb']:>9.1f}"
        )
    print()

## 2. Trade-offs, all configurations overlaid

Colour = method, marker = dataset (circle COCO, square Zappos), fill =
model (filled ViT-B/16, hollow ResNet18).

In [ ]:
colors = {
    "Exact FlatIP": "tab:gray",
    "HNSW": "tab:blue",
    "IVF-PQ": "tab:orange",
    "LSH": "tab:green",
}
markers = {"coco": "o", "zappos": "s"}
line_styles = {"vit": "-", "resnet": "--"}

def scatter(ax, run, x, y, color, size=60, **kwargs):
    filled = run["model"] == "vit"
    ax.scatter(x, y, marker=markers[run["dataset"]], s=size,
               facecolors=color if filled else "none", edgecolors=color,
               linewidths=1.5, alpha=0.85, **kwargs)

def style_legend(include_methods=True):
    handles = []
    if include_methods:
        handles += [plt.Line2D([], [], marker="o", linestyle="", color=c, label=a)
                    for a, c in colors.items()]
    handles += [
        plt.Line2D([], [], marker="o", linestyle="", color="black", label="COCO"),
        plt.Line2D([], [], marker="s", linestyle="", color="black", label="Zappos"),
        plt.Line2D([], [], marker="o", linestyle="", color="black", label="ViT-B/16 (filled)"),
        plt.Line2D([], [], marker="o", linestyle="", markerfacecolor="none",
                   color="black", label="ResNet18 (hollow)"),
    ]
    return handles

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
panels = [
    (axes[0], "latency_ms_per_query", "recall_at_10", "Accuracy vs speed", "Recall@10"),
    (axes[1], "index_size_mb", "recall_at_10", "Accuracy vs memory", "Recall@10"),
    (axes[2], "latency_ms_per_query", "map_at_10", "Ranking quality vs speed", "mAP@10"),
]
for key, run in runs.items():
    for row in run["rows"]:
        for ax, x_field, y_field, _, _ in panels:
            if row.get(y_field) is None:
                continue
            scatter(ax, run, row[x_field], row[y_field], colors[row["algorithm"]])
for ax, x_field, _, title, ylabel in panels:
    ax.set_xlabel("index size (MB, log)" if x_field == "index_size_mb" else "query latency (ms, 1 thread, log)")
    ax.set_xscale("log")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
axes[0].legend(handles=style_legend(), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Parameter sweeps: one line per configuration (line style = model).
sweep_specs = [
    ("HNSW", "efSearch"),
    ("IVF-PQ", "nprobe"),
    ("LSH", "nbits"),
]
fig, axes = plt.subplots(len(sweep_specs), 3, figsize=(19, 4.2 * len(sweep_specs)))
for i, (algo, param) in enumerate(sweep_specs):
    for key, run in runs.items():
        sub = [r for r in run["rows"] if r["algorithm"] == algo]
        x = [r["parameters"][param] for r in sub]
        style = dict(marker=markers[run["dataset"]], linestyle=line_styles[run["model"]],
                     label=run["label"])
        axes[i][0].plot(x, [r["recall_at_10"] for r in sub], **style)
        axes[i][1].plot(x, [r.get("map_at_10", np.nan) for r in sub], **style)
        axes[i][2].plot(x, [r["latency_ms_per_query"] for r in sub], **style)
    for ax, ylabel in zip(axes[i], ["Recall@10", "mAP@10", "ms/query"]):
        ax.set_xlabel(param)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{algo}: {ylabel} vs {param}")
        ax.set_xscale("log", base=2)
        ax.grid(alpha=0.3)
    axes[i][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Cross-configuration summary

Raw latencies depend on gallery size and vector dimension, so they are
**not** directly comparable across configurations. *Speed-up* = exact
latency / method latency within the same configuration; *compression* =
exact index size / method index size.

In [ ]:
def pick(results, algo, **params):
    for r in results:
        if r["algorithm"] == algo and all(
            r["parameters"].get(k) == v for k, v in params.items()
        ):
            return r
    return None

picks = [
    ("Exact FlatIP", "Exact FlatIP", {}),
    ("HNSW efSearch=16", "HNSW", {"efSearch": 16}),
    ("HNSW efSearch=64", "HNSW", {"efSearch": 64}),
    ("HNSW efSearch=128", "HNSW", {"efSearch": 128}),
    ("IVF-PQ nprobe=4", "IVF-PQ", {"nprobe": 4}),
    ("IVF-PQ nprobe=32", "IVF-PQ", {"nprobe": 32}),
    ("LSH nbits=256", "LSH", {"nbits": 256}),
    ("LSH nbits=512", "LSH", {"nbits": 512}),
]

group = f"{'R@10':>6}{'mAP':>7}{'speedup':>8}{'compr':>7}"
width = len(group)
print(f"{'':<18}" + " | ".join(f"{run['label']:^{width}}" for run in runs.values()))
print(f"{'setting':<18}" + " | ".join(group for _ in runs))
print("-" * (18 + (width + 3) * len(runs)))
for label, algo, params in picks:
    parts = []
    for run in runs.values():
        exact = pick(run["rows"], "Exact FlatIP")
        r = pick(run["rows"], algo, **params)
        if r is None:
            parts.append(f"{'n/a':>{width}}")
            continue
        parts.append(
            f"{r['recall_at_10']:>6.3f}{metric(r, 'map_at_10', 7)}"
            f"{exact['latency_ms_per_query'] / r['latency_ms_per_query']:>7.1f}x"
            f"{exact['index_size_mb'] / r['index_size_mb']:>6.1f}x"
        )
    print(f"{label:<18}" + " | ".join(parts))

# Speed-up vs recall on one normalised axis.
fig, ax = plt.subplots(figsize=(11, 6))
for key, run in runs.items():
    exact = pick(run["rows"], "Exact FlatIP")
    for r in run["rows"]:
        scatter(ax, run, exact["latency_ms_per_query"] / r["latency_ms_per_query"],
                r["recall_at_10"], colors[r["algorithm"]], size=75)
ax.axvline(1.0, color="tab:gray", linestyle="--", alpha=0.6)
ax.set_xscale("log")
ax.set_xlabel("speed-up vs exact search in the same configuration (log)")
ax.set_ylabel("Recall@10")
ax.set_title("Normalised speed-accuracy across all four configurations")
ax.legend(handles=style_legend(), fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Model vs dataset effects

The 2x2 grid separates the two factors:

- **model effect** = ViT-B/16 minus ResNet18 on the same dataset,
- **dataset effect** = COCO minus Zappos with the same model.

Positive values favour the first-named side. A method whose numbers move
mostly along one axis is mainly sensitive to that factor (e.g. vector
dimension for IVF-PQ / LSH, gallery size for speed-ups).

In [ ]:
grid_keys = ["vit-coco", "resnet-coco", "vit-zappos", "resnet-zappos"]
have_grid = all(k in runs for k in grid_keys)

def grid_value(key, label_algo_params, field):
    _, algo, params = label_algo_params
    r = pick(runs[key]["rows"], algo, **params)
    return np.nan if r is None or r.get(field) is None else r[field]

def speedup(key, label_algo_params):
    _, algo, params = label_algo_params
    exact = pick(runs[key]["rows"], "Exact FlatIP")
    r = pick(runs[key]["rows"], algo, **params)
    return np.nan if r is None else exact["latency_ms_per_query"] / r["latency_ms_per_query"]

if not have_grid:
    print("skipped: needs results for all four configurations")
else:
    for title, getter in [
        ("Recall@10", lambda k, p: grid_value(k, p, "recall_at_10")),
        ("mAP@10", lambda k, p: grid_value(k, p, "map_at_10")),
        ("speed-up vs exact", speedup),
    ]:
        print(f"=== {title} ===")
        print(f"{'setting':<18}{'ViT/COCO':>10}{'Res/COCO':>10}{'ViT/Zap':>10}{'Res/Zap':>10}"
              f" | {'model:COCO':>11}{'model:Zap':>10} | {'data:ViT':>9}{'data:Res':>9}")
        for p in picks[1:]:
            v = {k: getter(k, p) for k in grid_keys}
            print(
                f"{p[0]:<18}" + "".join(f"{v[k]:>10.3f}" for k in grid_keys)
                + f" | {v['vit-coco'] - v['resnet-coco']:>+11.3f}{v['vit-zappos'] - v['resnet-zappos']:>+10.3f}"
                + f" | {v['vit-coco'] - v['vit-zappos']:>+9.3f}{v['resnet-coco'] - v['resnet-zappos']:>+9.3f}"
            )
        print()

    # Grouped bars: each method setting across the 2x2 grid.
    fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
    bar_picks = picks[1:]
    x = np.arange(len(bar_picks))
    width = 0.2
    bar_colors = {"vit-coco": "#1f77b4", "resnet-coco": "#aec7e8",
                  "vit-zappos": "#d62728", "resnet-zappos": "#ff9896"}
    for ax, field, title in [(axes[0], "recall_at_10", "Recall@10"), (axes[1], "map_at_10", "mAP@10")]:
        for j, key in enumerate(grid_keys):
            values = [grid_value(key, p, field) for p in bar_picks]
            ax.bar(x + (j - 1.5) * width, values, width, label=runs[key]["label"],
                   color=bar_colors[key])
        ax.set_xticks(x)
        ax.set_xticklabels([p[0] for p in bar_picks], rotation=30, ha="right", fontsize=8)
        ax.set_ylabel(title)
        ax.set_title(f"{title} across the model x dataset grid")
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3, axis="y")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# How much do the two models agree? Overlap of the exact top-K neighbour
# sets of ViT and ResNet for the same query (same dataset, same queries).
def model_agreement(dataset, k):
    a, b = details.get(f"vit-{dataset}"), details.get(f"resnet-{dataset}")
    if a is None or b is None:
        return None
    overlap = [
        len(set(a["exact_top20"][q, :k].tolist()) & set(b["exact_top20"][q, :k].tolist())) / k
        for q in range(len(a["query_rows"]))
    ]
    return float(np.mean(overlap))

print("Exact-search neighbour overlap, ViT-B/16 vs ResNet18 (same queries):")
for dataset in ["coco", "zappos"]:
    values = [model_agreement(dataset, k) for k in (5, 10, 20)]
    if values[0] is None:
        print(f"  {DATASETS[dataset]['label']}: skipped (needs per-query files for both models)")
        continue
    print(f"  {DATASETS[dataset]['label']:<7} @5 {values[0]:.3f} | @10 {values[1]:.3f} | @20 {values[2]:.3f}")

## 5. Same query: both models, every method

One grid per query: the rows are the exact top-K of **both models** on the
same query, followed by the ANN settings of `GRID_MODEL`. ANN rows mark
`OK`/`MISS` against the `GRID_MODEL` exact top-K; the other model's exact
row marks `SAME`/`DIFF` - whether `GRID_MODEL`'s exact search also found
that image - which shows how differently the two models see similarity.

In [ ]:
def top_k(key, setting_key, k):
    if setting_key == "exact":
        return details[key]["exact_top20"][:, :k]
    return np.load(runs[key]["per_query_dir"] / f"{setting_key}_top20.npy")[:, :k]

def query_position(key, query_id):
    matches = np.where(details[key]["query_ids"] == int(query_id))[0]
    return int(matches[0]) if len(matches) else None

def to_image_id(key, gallery_row):
    # gallery row -> original embedding row -> image id
    d = details[key]
    return int(d["ids"][d["gallery_rows"][int(gallery_row)]])

def exact_id_set(key, position, k):
    return {to_image_id(key, row) for row in details[key]["exact_top20"][position, :k]}

def per_query_recall(key, setting_key, position, k):
    found = {to_image_id(key, row) for row in top_k(key, setting_key, k)[position] if row >= 0}
    return len(found & exact_id_set(key, position, k)) / k

def per_query_ap(key, setting_key, position, k):
    # AP@k against the exact top-k (same definition as benchmark_ann.map_at_k).
    reference = exact_id_set(key, position, k)
    hits, precision_sum = 0, 0.0
    for rank, row in enumerate(top_k(key, setting_key, k)[position], start=1):
        if row >= 0 and to_image_id(key, row) in reference:
            hits += 1
            precision_sum += hits / rank
    return precision_sum / k

def compare_query(dataset, query_id):
    grid_key = f"{GRID_MODEL}-{dataset}"
    other_model = "resnet" if GRID_MODEL == "vit" else "vit"
    other_key = f"{other_model}-{dataset}"
    if grid_key not in details or other_key not in details:
        print(f"{dataset}: needs per-query files for both models")
        return
    position = query_position(grid_key, query_id)
    if position is None:
        print(f"{dataset}: query {query_id} is not in the benchmark query set")
        return

    img = images[dataset]
    query_image = Image.open(img["path"](query_id)).convert("RGB")
    reference = exact_id_set(grid_key, position, K_DISPLAY)

    rows = [(grid_key, "exact", f"Exact ({MODELS[GRID_MODEL]['label']})"),
            (other_key, "exact", f"Exact ({MODELS[other_model]['label']})")]
    rows += [(grid_key, s, f"{label} ({MODELS[GRID_MODEL]['label']})") for s, label in SHOW_SETTINGS.items()]

    fig, axes = plt.subplots(len(rows), K_DISPLAY + 1,
                             figsize=(2.1 * (K_DISPLAY + 1), 3.0 * len(rows)))
    for r, (key, setting_key, label) in enumerate(rows):
        d = details[key]
        query_vector = np.asarray(d["embeddings"][int(d["query_rows"][position])])
        title = f"{DATASETS[dataset]['label']} query {query_id}\n{img['label'](query_id)}\n{label}"
        if setting_key != "exact":
            title += (f"\nrecall@{K_DISPLAY} {per_query_recall(key, setting_key, position, K_DISPLAY):.2f}"
                      f" | AP {per_query_ap(key, setting_key, position, K_DISPLAY):.2f}")
        elif key == other_key:
            agree = len(exact_id_set(key, position, K_DISPLAY) & reference) / K_DISPLAY
            title += f"\nagreement with {MODELS[GRID_MODEL]['label']}: {agree:.2f}"
        axes[r][0].imshow(query_image)
        axes[r][0].set_title(title, fontsize=8)
        axes[r][0].axis("off")

        for c, gallery_row in enumerate(top_k(key, setting_key, K_DISPLAY)[position], start=1):
            ax = axes[r][c]
            if gallery_row < 0:
                # FAISS pads with -1 when fewer than K results exist.
                ax.set_facecolor("lightgray")
                ax.set_title("not\nreturned", fontsize=8)
                ax.set_xticks([]); ax.set_yticks([])
                continue
            result_id = to_image_id(key, gallery_row)
            result_vector = np.asarray(d["embeddings"][int(d["gallery_rows"][gallery_row])])
            score = float(result_vector @ query_vector)  # cosine (normalized)
            hit = result_id in reference
            if key == other_key:
                tag = "SAME" if hit else "DIFF"
            else:
                tag = "OK" if hit else "MISS"
            ax.imshow(Image.open(img["path"](result_id)).convert("RGB"))
            ax.set_title(f"#{c} {tag}\n{img['label'](result_id)}\nid {result_id} | {score:.3f}",
                         fontsize=7, color="black" if hit else "red")
            ax.axis("off")

    plt.tight_layout()
    if SAVE_IMAGES:
        out_dir = runs[grid_key]["output_dir"]
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"query_{query_id}_methods.png"
        plt.savefig(out_path, dpi=110, bbox_inches="tight")
        print("saved:", out_path)
    plt.show()

for dataset in ["coco", "zappos"]:
    grid_key = f"{GRID_MODEL}-{dataset}"
    if grid_key not in details:
        continue
    if QUERY_IDS.get(dataset) is None:
        QUERY_IDS[dataset] = details[grid_key]["query_ids"][:3].tolist()
    if images.get(dataset) is None:
        print(f"{DATASETS[dataset]['label']}: images not found - grids skipped")
        continue
    for query_id in QUERY_IDS[dataset]:
        compare_query(dataset, query_id)

### Per-query recall and AP across every setting

Recall@K and AP@K (the per-query term of mAP@K) for the queries chosen
above, per configuration.

In [ ]:
def all_settings(key):
    return [
        p.name.replace("_top20.npy", "")
        for p in sorted(runs[key]["per_query_dir"].glob("*_top20.npy"))
        if p.name != "exact_top20.npy"
    ]

for key in details:
    dataset = runs[key]["dataset"]
    if not QUERY_IDS.get(dataset):
        continue
    columns = all_settings(key)
    for name, fn in [(f"recall@{K_DISPLAY}", per_query_recall), (f"AP@{K_DISPLAY}", per_query_ap)]:
        print(f"{runs[key]['label']} - {name}")
        print(f"{'query id':<10}" + "".join(f"{c:>12}" for c in columns))
        for query_id in QUERY_IDS[dataset]:
            position = query_position(key, query_id)
            if position is None:
                continue
            values = [fn(key, s, position, K_DISPLAY) for s in columns]
            print(f"{query_id:<10}" + "".join(f"{v:>12.2f}" for v in values))
        print()

## 6. Zappos category-level retrieval quality, both models

UT-Zap50K images carry labels from their folder structure: category
(`Boots / Sandals / Shoes / Slippers`) and subcategory (e.g. `Ankle`,
`Oxfords`). This measures how many of a method's top-10 results share the
**query's** category / subcategory, averaged over the 1,000 queries. It
answers a question vector recall cannot: which *model* retrieves more
semantically similar shoes, and how much approximate search costs in
those terms. COCO unlabeled2017 has no labels, so this is Zappos only.

In [ ]:
def label_precision(key, setting_key, labels, k=10):
    d = details[key]
    result_rows = top_k(key, setting_key, k)
    query_labels = labels[d["ids"][d["query_rows"]]][:, None]
    result_labels = labels[d["ids"][d["gallery_rows"][result_rows]]]
    return float((result_labels == query_labels)[result_rows >= 0].mean())

def row_for(key, setting_key):
    for r in runs[key]["rows"]:
        stem = {
            "HNSW": lambda p: f"hnsw_ef{p['efSearch']}",
            "IVF-PQ": lambda p: f"ivfpq_np{p['nprobe']}",
            "LSH": lambda p: f"lsh_nb{p['nbits']}",
        }.get(r["algorithm"])
        if stem and stem(r["parameters"]) == setting_key:
            return r
    return None

zap_keys = [k for k in ("vit-zappos", "resnet-zappos") if k in details]
if images.get("zappos") is None or not zap_keys:
    print("skipped: needs Zappos per-query files, embeddings and images")
else:
    cats, subs = images["zappos"]["categories"], images["zappos"]["subcategories"]
    header = f"{'setting':<22}" + "".join(
        f"{runs[k]['label'] + ' cat':>24}{'subcat':>8}" for k in zap_keys)
    print(f"Category / subcategory precision@10 ({len(details[zap_keys[0]]['query_rows'])} queries)")
    print(header)
    print("-" * len(header))
    for setting_key in ["exact"] + all_settings(zap_keys[0]):
        if setting_key == "exact":
            name = "Exact FlatIP"
        else:
            row = row_for(zap_keys[0], setting_key)
            name = f"{row['algorithm']} {setting_str(row)}"
        print(f"{name:<22}" + "".join(
            f"{label_precision(k, setting_key, cats):>24.3f}{label_precision(k, setting_key, subs):>8.3f}"
            for k in zap_keys))

## 7. Sanity checks

In [ ]:
for key, run in runs.items():
    exact_metrics = next(r for r in run["rows"] if r["algorithm"] == "Exact FlatIP")
    assert abs(exact_metrics["recall_at_10"] - 1.0) < 1e-9, f"{key}: exact recall@10 must be 1.0"
    for k in run["metadata"].get("metric_ks", []):
        assert abs(exact_metrics[f"precision_at_{k}"] - 1.0) < 1e-9, f"{key}: exact P@{k} must be 1.0"
        assert abs(exact_metrics[f"map_at_{k}"] - 1.0) < 1e-9, f"{key}: exact mAP@{k} must be 1.0"
        for r in run["rows"]:
            # Exact top-K and method top-K both hold K items, so P@K == R@K.
            assert abs(r[f"precision_at_{k}"] - r[f"recall_at_{k}"]) < 1e-9, f"{key}: P@{k} != R@{k}"
            assert r[f"map_at_{k}"] <= r[f"recall_at_{k}"] + 1e-9, f"{key}: mAP@{k} > R@{k}"

    if key not in details:
        print(f"{run['label']}: aggregate checks passed (per-query checks skipped)")
        continue

    d = details[key]
    num_queries = run["metadata"]["num_queries"]
    k_gt = run["metadata"]["k_gt"]
    gallery_size = run["metadata"]["gallery_size"]
    for path in sorted(run["per_query_dir"].glob("*_top20.npy")):
        arr = np.load(path)
        assert arr.shape == (num_queries, k_gt), f"{key}: bad shape in {path.name}: {arr.shape}"
        assert arr.min() >= -1 and arr.max() < gallery_size, f"{key}: out-of-range rows in {path.name}"

    # Held-out split: a query image must never appear in its own results.
    for position in range(num_queries):
        query_id = int(d["query_ids"][position])
        result_ids = {to_image_id(key, row) for row in d["exact_top20"][position] if row >= 0}
        assert query_id not in result_ids, f"{key}: query {query_id} leaked into gallery"
    print(f"{run['label']}: ALL CHECKS PASSED")

# Both models on a dataset must be searched with the identical queries.
for dataset in ["coco", "zappos"]:
    a, b = details.get(f"vit-{dataset}"), details.get(f"resnet-{dataset}")
    if a is None or b is None:
        continue
    assert np.array_equal(a["query_ids"], b["query_ids"]), f"{dataset}: query sets differ between models"
    assert np.array_equal(a["ids"], b["ids"]), f"{dataset}: image id order differs between models"
    print(f"{DATASETS[dataset]['label']}: both models use the identical {len(a['query_ids'])} queries")

## How to adjust

- **Fewer configurations**: edit `CONFIG_KEYS` in the config cell.
- **Other queries**: set `QUERY_IDS["coco"]` / `QUERY_IDS["zappos"]` to ids
  printed in the setup cell, then re-run section 5.
- **Which model's ANN rows are shown**: `GRID_MODEL = "vit"` or `"resnet"`.
- **Other settings side by side**: replace keys in `SHOW_SETTINGS` with any
  listed setting (e.g. `"hnsw_ef256"`).
- **More displayed results**: raise `K_DISPLAY` (up to 20, the saved
  ground-truth depth).
- The notebook only reads benchmark outputs; it never recomputes FAISS or
  the embedding models.